# Foundation-model copula comparison

This notebook fits three `FoundationModelBicop` variants to the same sample:

1. TabPFN with native-distribution recovery,
2. TabPFN with quantile-inversion recovery, and
3. TabICL with quantile-inversion recovery.

They are compared with pyvinecopulib's transformation local likelihood (`tll`) estimator and the known Clayton data-generating copula. Model weights may be downloaded on first use; install the `foundation-models` and `interactive` extras first.

In [ ]:
from time import perf_counter

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pyvinecopulib as pv
from dotenv import load_dotenv

from npcc import (
  FoundationModelBicop,
  Recovery,
  SupportTransform,
  TabICLConfig,
  TabPFNConfig,
)

load_dotenv()  # Loads TABPFN_TOKEN from .env when needed.
SEED = 317

## Simulate one shared training sample

For a Clayton copula, $\tau = \theta/(\theta+2)$, so $\theta=2$ gives Kendall's $\tau=0.5$. The sample size is above TabICLv2's documented lower pretraining boundary.

In [ ]:
n_train = 1_000
truth = pv.Bicop(
  family=pv.BicopFamily.clayton,
  parameters=np.array([[2.0]]),
)
uv_train = truth.simulate(n_train, seeds=[SEED, SEED + 1, SEED + 2])
u_train, v_train = uv_train[:, 0], uv_train[:, 1]
uv_train[:5]

## Fit the pyvinecopulib benchmark

The family set is restricted to `tll`, making this a nonparametric benchmark rather than a model-selection exercise over parametric families.

In [ ]:
t0 = perf_counter()
tll = pv.Bicop.from_data(
  uv_train,
  controls=pv.FitControlsBicop(family_set=[pv.BicopFamily.tll]),
)
tll_fit_seconds = perf_counter() - t0
print(f"TLL fit: {tll_fit_seconds:.2f} s; estimated tau={tll.tau:.3f}")

## Configure and fit the three foundation-model estimators

Provider and recovery are explicit. All models use the same support transform, training observations, and base random seed. TabPFN supports both recovery paths; TabICL supports quantile inversion only.

In [ ]:
models = {
  "TabPFN native": FoundationModelBicop(
    provider="tabpfn",
    recovery=Recovery.NATIVE_DISTRIBUTION,
    provider_config=TabPFNConfig(model_version="v3"),
    transform=SupportTransform.LOGIT,
    random_state=SEED,
  ),
  "TabPFN quantile": FoundationModelBicop(
    provider="tabpfn",
    recovery=Recovery.QUANTILE_INVERSION,
    provider_config=TabPFNConfig(model_version="v3"),
    transform=SupportTransform.LOGIT,
    random_state=SEED,
  ),
  "TabICL quantile": FoundationModelBicop(
    provider="tabicl",
    recovery=Recovery.QUANTILE_INVERSION,
    provider_config=TabICLConfig(),
    transform=SupportTransform.LOGIT,
    random_state=SEED,
  ),
}

fit_seconds = {}
for name, model in models.items():
  t0 = perf_counter()
  model.fit(u_train, v_train)
  fit_seconds[name] = perf_counter() - t0
  print(f"{name:18s}: {fit_seconds[name]:.2f} s")

## Evaluate density accuracy on a common grid

The grid stays away from the open copula boundary. IAE, ISE, and KL are computed by two-dimensional trapezoidal integration. Densities are clipped only inside the logarithm used for KL.

In [ ]:
grid_size = 60
axis = np.linspace(0.01, 0.99, grid_size)
u_mesh, v_mesh = np.meshgrid(axis, axis, indexing="ij")
uv_grid = np.column_stack([u_mesh.ravel(), v_mesh.ravel()])

densities = {
  "Truth": truth.pdf(uv_grid).reshape(grid_size, grid_size),
  "pyvinecopulib TLL": tll.pdf(uv_grid).reshape(grid_size, grid_size),
}
evaluation_seconds = {}
for name, model in models.items():
  t0 = perf_counter()
  densities[name] = np.asarray(model.pdf_grid(axis, axis))
  evaluation_seconds[name] = perf_counter() - t0

def integrate_2d(values):
  return float(np.trapezoid(np.trapezoid(values, axis=1, x=axis), x=axis))

truth_density = densities["Truth"]
rows = []
for name, estimate in densities.items():
  if name == "Truth":
    continue
  difference = estimate - truth_density
  safe_estimate = np.clip(estimate, 1e-12, None)
  safe_truth = np.clip(truth_density, 1e-12, None)
  rows.append({
    "estimator": name,
    "fit_seconds": tll_fit_seconds if name == "pyvinecopulib TLL" else fit_seconds[name],
    "grid_seconds": np.nan if name == "pyvinecopulib TLL" else evaluation_seconds[name],
    "IAE": integrate_2d(np.abs(difference)),
    "ISE": integrate_2d(difference**2),
    "KL": integrate_2d(safe_truth * np.log(safe_truth / safe_estimate)),
    "density_integral": integrate_2d(estimate),
  })
comparison = pd.DataFrame(rows).sort_values("ISE")
comparison

## Compare density surfaces

All panels use the same contour levels, determined jointly across the truth and estimates.

In [ ]:
panel_names = list(densities)
positive = np.concatenate([density.ravel() for density in densities.values()])
levels = np.linspace(0.0, np.quantile(positive, 0.98), 16)
fig, axes = plt.subplots(2, 3, figsize=(15, 9), constrained_layout=True)

for ax, name in zip(axes.flat, panel_names, strict=False):
  contour = ax.contourf(axis, axis, densities[name].T, levels=levels, cmap="viridis", extend="max")
  ax.set(title=name, xlabel="u", ylabel="v", aspect="equal")

for ax in axes.flat[len(panel_names):]:
  ax.set_visible(False)

fig.colorbar(contour, ax=axes, label="copula density", shrink=0.85)
plt.show()

## Pointwise error maps

These panels show signed density error relative to the known Clayton copula on a shared symmetric color scale.

In [ ]:
estimate_names = [name for name in densities if name != "Truth"]
errors = {name: densities[name] - truth_density for name in estimate_names}
error_limit = np.quantile(np.abs(np.concatenate([e.ravel() for e in errors.values()])), 0.98)
fig, axes = plt.subplots(1, 4, figsize=(18, 4), constrained_layout=True)

for ax, name in zip(axes, estimate_names, strict=True):
  image = ax.contourf(axis, axis, errors[name].T, levels=21, cmap="coolwarm", vmin=-error_limit, vmax=error_limit)
  ax.set(title=name, xlabel="u", ylabel="v", aspect="equal")

fig.colorbar(image, ax=axes, label="estimated density - truth", shrink=0.85)
plt.show()